In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import arya
import pandas as pd

In [ ]:
import ana_utils

In [ ]:
from astropy import units as u

In [ ]:
from dataclasses import dataclass

In [ ]:
from astropy.coordinates import SkyCoord

In [ ]:
obj = "yasone1"

In [ ]:
cat = ana_utils.read_catalogue(obj, filter_bad=False)
ana_utils.add_flux_param(cat)
cat["INDEX"] = np.arange(len(cat))

In [ ]:
obs_props = ana_utils.get_obs_props(obj)
A_g, A_r, A_i = ana_utils.get_extinction(obs_props["A_V"])

In [ ]:
coord0 = SkyCoord(obs_props["ra"], obs_props["dec"], unit=u.degree)

In [ ]:
plt.scatter(cat["xi"], cat["eta"], s=1, lw=0, color="k")
plt.xlabel("xi / arcmin")
plt.ylabel("eta / arcmin")
plt.title("no selection")
plt.gca().invert_xaxis()
plt.gca().set_aspect(1)

## Extinction

In [ ]:
from dustmaps.sfd import SFDQuery
from dustmaps.bayestar import BayestarWebQuery
from dustmaps.planck import PlanckQuery

In [ ]:
coords = ana_utils.to_coords(cat)

In [ ]:
def plot_dustmap(dustmap, distance, xymax=6, N=100):

    ra = np.linspace(-xymax, xymax, N) / 60 / np.cos(np.deg2rad(obs_props["dec"])) + obs_props["ra"]
    dec = np.linspace(-xymax, xymax, N) / 60 + obs_props["dec"]
    ra, dec = np.meshgrid(ra, dec)
    coords = SkyCoord(ra*u.deg, dec*u.deg, distance=distance*u.kpc)


    Av = dustmap(coords)
    plt.imshow(Av, origin="lower", extent=(np.min(ra), np.max(ra), np.min(dec), np.max(dec))
              )
    plt.colorbar()
    plt.xlabel("ra / deg")
    plt.ylabel("dec / deg")

In [ ]:
ra0 = obs_props["ra"]
dec0 = obs_props["dec"]
ra0, dec0

In [ ]:
coord0.ra.hms, coord0.dec.dms

In [ ]:
plot_dustmap(SFDQuery(), 12, xymax=3*60)

xl, xh = ra0 - 3/60 / np.cos(np.deg2rad(dec0)), ra0 + 3/60 / np.cos(np.deg2rad(dec0))
yl, yh = dec0 - 3/60, dec0 + 3/60
# plt.plot([xl, xh, xh, xl, xl], [yl, yl, yh, yh, yl])

xl, xh = np.min(cat["R_ALPHA_J2000"]), np.max(cat["R_ALPHA_J2000"])
yl, yh = np.min(cat["R_DELTA_J2000"]), np.max(cat["R_DELTA_J2000"])

plt.plot([xl, xh, xh, xl, xl], [yl, yl, yh, yh, yl], lw=1)


In [ ]:
ra0, dec0

In [ ]:
Ebv = SFDQuery()(coord0)
Ebv, 3.1*Ebv

In [ ]:
plot_dustmap(SFDQuery(), 12)

xl, xh = np.min(cat["R_ALPHA_J2000"]), np.max(cat["R_ALPHA_J2000"])
yl, yh = np.min(cat["R_DELTA_J2000"]), np.max(cat["R_DELTA_J2000"])

plt.plot([xl, xh, xh, xl, xl], [yl, yl, yh, yh, yl], lw=1)


The above plots show the SFD extinction maps around our region. The rectangle is the bounds of the catalog. Over this region, the extinction only typically varies by 0.01 dex, so extinction correction appears to be a smaller effect here.

In [ ]:
# deredden magnitudes

coords = ana_utils.to_coords(cat)
A_V = 3.1 * SFDQuery()(coords)
A_g, A_r, A_i = ana_utils.get_extinction(A_V)

In [ ]:
np.median(A_g), np.median(A_r), np.median(A_i)

In [ ]:
cat["G_MAG"] -= A_g
cat["R_MAG"] -= A_r
cat["I_MAG"] -= A_i


## Mangitude setup

Since only the default magnitude columns (G_MAG, R_MAG, and I_MAG) are calibrated, 
we simply use these columns to calibrate other magnitude systems. The other columns do not
have a zeropoint offset. For simplicity, I use stars of magnitudes 19 to 22 

In [ ]:
def calibrate_mag_col(cat, col):
    for filt in ["G", "R", "I"]:
        filt_good = cat[f"{filt}_MAG"] > 19
        filt_good &= cat[f"{filt}_MAG"] < 22
        filt_good &= ~cat[f"{filt}_MAG"].mask
        filt_good &= cat[f"{filt}_FLAGS"] == 0
        if hasattr(cat[f"{filt}_{col}"], "mask"):
            filt_good &= ~cat[f"{filt}_{col}"].mask
            
        residual = np.median((cat[f"{filt}_{col}"] - cat[f"{filt}_MAG"])[filt_good])

        print(residual)
        cat[f"{filt}_{col}"] -= residual

In [ ]:
mag_choices = ["MAG_APER_0", "MAG_APER_1", "MAG_APER_2", "MAG_APER_3", "MAG_APER_4", "MAG_APER_5", "MAG_APER_6", "MAG_AUTO", "MAG_PSF"]

In [ ]:
for mag in mag_choices:
    calibrate_mag_col(cat, mag)

In [ ]:
col = "MAG_PSF"
for filt in ["G", "R", "I"]:
    plt.scatter(cat[f"{filt}_MAG"], cat[f"{filt}_{col}"] - cat[f"{filt}_MAG"], s=1, lw=0, color="k", alpha=0.3)
    plt.axhline(0, color="C1")
    plt.ylim(-1, 1)
    plt.xlim(12, 30)
    plt.xlabel(f"default magnitude {filt}")
    plt.ylabel(f"calibrated windowed magnitude - default")
    plt.show()

The above plots show an example of this calibration for the Source Extractor windowed magnitudes. In general, this naive method works reasonably well

## Star Galaxy Seperation

For star-galaxy seperation, we use the difference in the 3px and 5px aperture magnitudes. 
3px is about the typical FWHM of good point-like sources, so sources with a larger increase in the magnitudes between 3px and 5px
are likely galaxies. However, this method is strongly biased by poor-background behaviour (nearby bright sources).

In [ ]:
aperture_radii = [1, 2, 3, 5, 7, 10, 20]

In [ ]:
aperture_radii[2]

In [ ]:
cat["MAG_23"] =  cat["R_MAG_APER_2"] - cat["R_MAG_APER_3"]

In [ ]:
from astropy.stats import sigma_clipped_stats

In [ ]:
def filter_not_bright_point(cat):
    filt_good = cat["R_FLUX_RADIUS"] < 3
    filt_good &= cat["R_MAG"] < 21
    filt_good &= cat["R_ELLIPTICITY"] < 0.3
    return filt_good

In [ ]:
def derive_threshhold(flux_param, cat, sigma=3):

    filt = filter_not_bright_point(cat)
    
    μ_flux, med_flux, std_flux = sigma_clipped_stats(flux_param[filt], stdfunc="mad_std", sigma=3)

    return med_flux + sigma*std_flux

In [ ]:
mag_diff_thresh = derive_threshhold(cat["MAG_23"], cat)

In [ ]:
cat_good = cat[filter_not_bright_point(cat)]

In [ ]:
def filt_finite(col):
    filt = np.isfinite(col)
    if hasattr(col, "mask"):
        filt &= ~col.mask

    return col[filt]
    

In [ ]:
[np.nanmedian(filt_finite(cat_good[f"{filt}_FWHM_IMAGE"])) for filt in ["G", "R", "I"]]

In [ ]:
[np.nanmedian(filt_finite(cat_good[f"{filt}_FWHM_WORLD"] * 3600)) for filt in ["G", "R", "I"]]

In [ ]:
plt.scatter(cat["R_MAG"], cat["MAG_23"],c=np.log10(cat["R_FWHM_IMAGE"]), vmin=0.3, vmax=1.3, s=1, alpha=1, lw=0)
plt.axhline(mag_diff_thresh, color="k")
plt.ylim(-1, 1)
plt.xlim(18, 27)
plt.xlabel("R mag")
plt.ylabel("R(3pix) - R(5pix) (mag)")
plt.colorbar(label="log fwhm / pix")

In [ ]:
ϵ = 0.003
χ = 3
sg_sep = cat["R_SPREAD_MODEL"] < np.sqrt(ϵ**2 + χ**2 * cat["R_SPREADERR_MODEL"]**2)


In [ ]:
plt.scatter(cat["R_SPREAD_MODEL"], cat["R_MAG"], c=sg_sep, s=1, lw=0)
plt.xlim(-0.05, 0.05)
plt.ylim(27, 16)

In [ ]:
plt.scatter(cat["R_MAG"], cat["MAG_23"],c=sg_sep, s=1, alpha=1, lw=0)
plt.axhline(mag_diff_thresh, color="k")
plt.ylim(-1, 1)
plt.xlim(18, 27)
plt.xlabel("R mag")
plt.ylabel("R(3pix) - R(5pix) (mag)")
plt.colorbar(label="log fwhm / pix")

# Bounds

We want to sample random positions which we can place complete circles. 
The below code uses a polyhedral region around the target field and excludes regions nearby the centre (where we measure),
nearby bright stars (for Yasone1) and nearby the edges.

In [ ]:
from shapely.geometry import Point
from shapely import Polygon

In [ ]:

def sample_point(poly):
    minx, miny, maxx, maxy = poly.bounds

    while True:
        x = random.uniform(minx, maxx)
        y = random.uniform(miny, maxy)
        p = Point(x, y)
        if poly.contains(p):
            return p

In [ ]:
def to_polygon(flat_coords):
    ra = flat_coords[::2]
    dec = flat_coords[1::2]
    coords = SkyCoord(ra, dec, unit=u.degree)
    xi, eta = ana_utils.to_tangent(coords, coord0)
    
    points = [p for p in zip(xi*60, eta*60)]
    return Polygon(points)

In [ ]:
if obj == "yasone1":
    region_mask = [265.442,13.208, 265.485,13.206, 265.487,13.194, 265.518,13.195, 265.534,13.214, 265.524,13.228, 265.508,13.240, 265.491,13.228, 265.456,13.218, 265.442,13.213]

    region_poly = [265.446,13.118, 265.591,13.120, 265.592,13.250, 265.440,13.248]
elif obj == "yasone2":
    region_poly = [262.276,6.370, 262.419,6.372, 262.420,6.502, 262.272,6.500]
    region_mask = None

elif obj == "yasone3":
    region_poly = [292.8815,-26.5033, 293.0386,-26.5005, 293.0401,-26.3710, 292.8771,-26.3716]
    region_mask = None
else:
    region_poly = None
    region_mask = None

In [ ]:
def get_sample_region(radius, mask=region_mask, poly=region_poly):
    safe_region = to_polygon(poly)
    safe_region = safe_region.buffer(-radius)

    if mask is not None:
        mask_poly = to_polygon(mask)
        safe_region = safe_region.difference(mask_poly.buffer(radius))

    centre = Point(0,0).buffer(2*radius)
    safe_region = safe_region.difference(centre)
    return safe_region

In [ ]:
safe_region = get_sample_region(40/60)

In [ ]:
def sample_from_poly(poly):
    minx, miny, maxx, maxy = poly.bounds

    while True:
        x = np.random.uniform(minx, maxx)
        y = np.random.uniform(miny, maxy)
        p = Point(x, y)
        if poly.contains(p):
            return p.x, p.y

In [ ]:
A_sampled = safe_region.area / (np.pi  * (40/60)**2)

In [ ]:
A_sampled

In [ ]:
points = [sample_from_poly(safe_region) for i in range(10000)]

In [ ]:
plt.scatter(cat["xi"], cat["eta"], s=1, c="k")
plt.scatter([p[0] for p in points], [p[1] for p in points], alpha=0.1)
plt.xlabel("xi")
plt.ylabel("eta")

The black points above show the full catalogue. The blue region shows where random samples are being drawn from

In [ ]:
tangent_bounds = to_polygon(region_poly).bounds

# Utilities

In [ ]:
def count_centre(cat, radius, xi0=0, eta0=0):
    return np.sum((cat["xi"] - xi0)**2 + (cat["eta"] - eta0)**2 < radius**2)

In [ ]:
def count_not_centre(cat, radius, xymax=3, xi0=0, eta0=0):
    filt = (cat["xi"] - xi0)**2 + (cat["eta"] - eta0)**2 >= radius**2
    filt &= cat["xi"] < xymax
    filt &= cat["xi"] > -xymax
    filt &= cat["eta"] < xymax
    filt &= cat["eta"] > -xymax

    return np.sum(filt)

In [ ]:
Ntot = np.sum((cat["xi"] > -3) & (cat["xi"] < 3) & (cat["eta"] < 3) & (cat["eta"] > -3))
Ncen = count_centre(cat, 40/60)

In [ ]:
cat_cen = ana_utils.select_centre(cat, ana_utils.get_coord0(obj), 40 * u.arcsec)
assert count_not_centre(cat, 40/60) == Ntot - Ncen
assert len(cat_cen) == Ncen

In [ ]:
count_centre(cat, 40/60)

In [ ]:
def counts_to_stats(Ntot, Ncen, area_scale):
    Ntot_normed = Ntot * area_scale
    Ncen_err = np.sqrt(Ncen)
    Ntot_normed_err = np.sqrt(Ntot) * area_scale

    num_err = np.sqrt(Ntot_normed_err**2 + Ncen_err**2)

    sigma = (Ncen - Ntot_normed) / Ntot_normed
    sigma_err = np.sqrt((Ncen_err / Ntot_normed)**2 + (Ncen/Ntot_normed**2 * Ntot_normed_err)**2)

    return sigma, sigma_err, Ntot, Ncen

In [ ]:
def inner_overdensity(cat, radius, xymax=3):
    area_tot = (2*xymax)**2
    area_cen = np.pi * radius**2
    area_scale = area_cen / (area_tot - area_cen)

    Ntot = count_not_centre(cat, radius, xymax=xymax)
    Ncen = count_centre(cat, radius)

    return counts_to_stats(Ntot, Ncen, area_scale)


In [ ]:
import read_iso

In [ ]:
isos_fe_h = ["m1.00", "m1.25", "m1.50", "m1.75", "m2.00", "m2.50", "m3.00"]

isonames = [f"../MIST/MIST_v1.2_vvcrit0.4_SDSSugriz/MIST_v1.2_feh_{iso_fe_h}_afe_p0.0_vvcrit0.4_SDSSugriz.iso.cmd" for iso_fe_h in isos_fe_h]
isochrones = {fe_h: read_iso.ISOCMD(name) for fe_h, name in zip(isos_fe_h, isonames)}


In [ ]:
def make_polygon(iso, mag_err, dm, A_b, A_r, iso_width=0.05, b="SDSS_g", r="SDSS_r", err_scale=1):
    filt = iso["phase"] < 3

    x = iso[b][filt].data - iso[r][filt].data + A_b - A_r
    y = iso[b][filt].data + dm + A_b

    x_min = x - err_scale * mag_err(y) - iso_width
    x_max = x + err_scale * mag_err(y) + iso_width

    return np.hstack([x_min, x_max[::-1], x_min[0]]), np.hstack([y, y[::-1], y[0]])

In [ ]:
gr_err = ana_utils.fit_err(cat["G_MAG"], cat["GR_ERR"])
ri_err = ana_utils.fit_err(cat["R_MAG"], cat["RI_ERR"])

In [ ]:
def select_gr(cat_gr, params):
    iso = isochrones[params.iso_fe_h][params.iso_log_age]

    A_g, A_r, A_i = get_extinction(params)
    x_poly_gr, y_poly_gr = make_polygon(iso, gr_err, params.dm, A_b=A_g, A_r=A_r, r="SDSS_r", b="SDSS_g", iso_width=params.iso_width, err_scale=params.err_scale)
    
    cmd_filt_gr = ana_utils.is_in_poly(cat_gr[params.gcol].data - cat_gr[params.rcol].data, cat_gr[params.gcol].data, x_poly_gr, y_poly_gr)

    return cmd_filt_gr



def select_ri(cat, params):
    iso = isochrones[params.iso_fe_h][params.iso_log_age]

    A_g, A_r, A_i = get_extinction(params)
    x_poly_gr, y_poly_gr = make_polygon(iso, ri_err, params.dm, 
                                        A_b=A_r, A_r=A_i, r="SDSS_i", b="SDSS_r", iso_width=params.iso_width, err_scale=params.err_scale)
    
    cmd_filt = ana_utils.is_in_poly(cat[params.rcol].data - cat[params.icol].data, cat[params.rcol].data, x_poly_gr, y_poly_gr)

    return cmd_filt

In [ ]:
@dataclass
class filter_params:
    r_cen: float = 25/60
    xymax: float = 3
    flags_max: int = 1024
    flags_weight_max: int = 2
    detection_sigma: float = 1.5
    A_V: float = 0
    dm: float = obs_props["dm"]
    iso_width: float = 0.2
    iso_fe_h: str = obs_props["iso_fe_h"]
    iso_log_age: float = np.round(np.log10(obs_props["iso_age"])*10) / 10
    err_scale: float = 1
    mode: str = "ri"
    gcol:str = "G_MAG"
    rcol: str = "R_MAG"
    icol: str = "I_MAG"
    A_i_extra: float = 0

    # star galaxy methods
    ellipticity_max: float = 1
    r_half_max: float = np.inf
    fwhm_max: float = np.inf
    r23_max: float = np.inf
    r23_max_sigma: float = np.nan
    


In [ ]:
filter_params()

In [ ]:
def filter_nans(col):
    filt = np.isfinite(col)
    if hasattr(col, "mask"):
        filt &= ~col.mask

    return filt


In [ ]:
def get_extinction(params):
    A_g, A_r, A_i = ana_utils.get_extinction(params.A_V)
    return A_g, A_r, A_i + params.A_i_extra


In [ ]:
def filter_catalog(cat, params):    
    filt = cat["R_FLAGS"] <= params.flags_max
    filt &= cat["R_FLAGS_WEIGHT"] <= params.flags_weight_max
    filt &= cat["R_SNR"] > params.detection_sigma
    filt &= cat["R_ELLIPTICITY"] <= params.ellipticity_max
    filt &= cat["R_FWHM_IMAGE"] <= params.fwhm_max
    filt &= cat["R_FLUX_RADIUS"] <= params.r_half_max

    if np.isfinite(params.r23_max_sigma):
        thresh_23 = derive_threshhold(cat["MAG_23"][filt], cat[filt])
    else:
        thresh_23 = params.r23_max
    filt &= cat["MAG_23"] < thresh_23
    
    if params.mode == "gr":
        filt &= select_gr(cat, params)
        filt &= filter_nans(cat[params.gcol])
        filt &= filter_nans(cat[params.rcol])
    elif params.mode == "ri":
        filt &= select_ri(cat, params)
        filt &= filter_nans(cat[params.icol])
        filt &= filter_nans(cat[params.rcol])
    elif params.mode == "gri":
        filt &= select_gr(cat, params)
        filt &= select_ri(cat, params)
        filt &= filter_nans(cat[params.gcol])
        filt &= filter_nans(cat[params.rcol])
        filt &= filter_nans(cat[params.icol])
    elif params.mode == "n":
        pass
    elif params.mode == "r":
        filt &= filter_nans(cat[params.rcol])
    else:
        raise Exception(f"mode not known {params.mode}")
        return None


    return cat[filt]


In [ ]:
from dataclasses import asdict

In [ ]:
from scipy.spatial import cKDTree

In [ ]:
def background_sample_counts(cat, radius, xymax=3, N=1000):
    points = cat[["xi", "eta"]].to_pandas().to_numpy()
    tree = cKDTree(points)

    # sampling radii from outside the central region but within xy max
    xi = np.random.uniform(-xymax+radius, -1.5*radius, N) * np.random.choice([1, -1], N)
    eta = np.random.uniform(-xymax+radius, -1.5*radius, N) * np.random.choice([1, -1], N)

    centres = np.column_stack([xi, eta])
    counts = tree.query_ball_point(centres, r=radius, return_length=True)
    return counts

In [ ]:
from astropy.stats import mad_std

In [ ]:
def catalog_stats(cat, 
        params
    ):

    cat_filtered = filter_catalog(cat,
        params
    )
    sigma, sigma_err, Ntot, Ncen = inner_overdensity(cat_filtered, params.r_cen, xymax=params.xymax)



    N_bkgs = background_sample_counts(cat_filtered, params.r_cen, xymax=params.xymax)
    
    area_tot = (2*params.xymax)**2
    area_cen = np.pi * params.r_cen**2
    area_scale = area_cen / (area_tot - area_cen)
    Nbkg_med = np.median(N_bkgs)
    Nbkg_std = mad_std(N_bkgs)

    delta = (Ncen - Nbkg_med) / Nbkg_med
    delta_err = np.sqrt((np.sqrt(Ncen) / Nbkg_med)**2 + (Ncen/Nbkg_med**2 * Nbkg_std)**2)

    pval = np.mean(N_bkgs > Ncen)

    return pd.DataFrame(dict(
        pval = pval,
        sigma = sigma, 
        sigma_err = sigma_err,
        delta = delta,
        delta_err = delta_err,
        Ntot = Ntot,
        Ntot_scaled = Ntot * area_scale,
        area_scale = area_scale,
        Ncen = Ncen,
        Nbkg_med = Nbkg_med,
        Nbkg_std = Nbkg_std,
        **asdict(params)
        
    ), index=[0])

In [ ]:
catalog_stats(cat, filter_params())

# Simple statistics (no variations)

In [ ]:
def rand_overdensity(cat, radius, safe_region=None):
    if safe_region is None:
        safe_region = get_sample_region(radius)
    xi, eta = sample_from_poly(safe_region)    

    Ncen = count_centre(cat, radius, xi0=xi, eta0=eta)
    return Ncen


In [ ]:
def plot_2d_counts(cat, radius, xymax=5, N=100, vmin=None, vmax=None, thresh=None):
    x = np.linspace(tangent_bounds[0], tangent_bounds[2], N)
    y = np.linspace(tangent_bounds[1], tangent_bounds[3], N)

    xgrid, ygrid = np.meshgrid(x, y)
    counts = np.zeros((N, N))

    for i in range(N):
        for j in range(N):
            counts[i, j] = count_centre(cat, radius, xi0=xgrid[i, j], eta0=ygrid[i, j])

    level_step = np.maximum(1, np.ceil(np.max(counts) / 15))

    p = plt.contourf(x, y, counts, levels=np.arange(np.min(counts), np.max(counts)+level_step, level_step), vmin=vmin, vmax=vmax)

    if thresh is not None:
        plt.contour(x, y, counts, levels=[thresh], color="k", lw=0.3)

    ax =  plt.gca()
    plt.colorbar(p, label="counts within circle", fraction=0.046, pad=0.04)
    plt.xlabel(r"$\xi$ / arcmin")
    plt.ylabel(r"$\eta$ / arcmin")

    circ = plt.Circle((0, 0), radius, color="k", lw=0.2, fill=False, alpha=1)
    ax.add_artist(circ)

    plt.scatter(0, 0, color="k", s=0.3, lw=0)
    ax.set_aspect(1)
    ax.invert_xaxis()
    return counts

In [ ]:
def plot_with_hist(cat, params, vmax=None, vmin=None):
    fig, axs = plt.subplots(2, 4, figsize=(8, 4))

    cat_filtered = filter_catalog(cat, params)
    results = catalog_stats(cat, params)
    safe_region = get_sample_region(params.r_cen)
    background_counts = [rand_overdensity(cat_filtered, params.r_cen, safe_region=safe_region) for i in range(10_000)]
    Ncen = count_centre(cat_filtered, params.r_cen)
    
    plot_riso(cat, params, results=results.iloc[0, :], fig=fig, axs=axs)
    

    plt.sca(axs[1][-1])
    grid_counts = plot_2d_counts(cat_filtered, params.r_cen, vmin=vmin, vmax=vmax, thresh=Ncen)



    plt.sca(axs[0][-1])
    plt.hist(background_counts, histtype="step", bins=np.arange(0, np.max(background_counts)))
    plt.hist(grid_counts.flatten(), histtype="step", bins=np.arange(0, np.max(grid_counts.flatten())))
    plt.gca().set_box_aspect(1)
    plt.xlabel("N within random circle")
    plt.ylabel("counts")
    p = np.mean(Ncen < background_counts)
    p2 = np.mean(Ncen < grid_counts)
    plt.text(0.9, 0.9, f"p = {p}", transform=plt.gca().transAxes, fontsize=8, ha="right", va="bottom", color="C1")
    plt.text(0.9, 0.85, f"p2 = {p2}", transform=plt.gca().transAxes, fontsize=8, ha="right", va="top", color="C2")
    
    plt.axvline(Ncen)



In [ ]:
import seaborn as sns

In [ ]:
def plot_riso(cat, params, xi=-1.5/60, eta=-1.5/60, ra0=obs_props["ra"], dec0=obs_props["dec"], results=None, fig=None, axs=None):

    cat_filtered = filter_catalog(cat, params)
    
    # xi = np.random.uniform(1.5 *params.r_cen, 3 - params.r_cen) / 60 * np.random.choice([-1, 1])
    # eta = np.random.uniform(1.5 * params.r_cen, 3 - params.r_cen) / 60 * np.random.choice([-1, 1])
    
    coord1 = SkyCoord(ra=ra0 + xi/np.cos(np.deg2rad(dec0)), dec=dec0 + eta, unit=u.deg)
    coord0 = SkyCoord(ra=ra0, dec=dec0, unit=u.deg)

    df_bkg = ana_utils.select_centre(cat_filtered, coord1, params.r_cen * u.arcmin)
    df_cen = ana_utils.select_centre(cat_filtered, coord0, params.r_cen * u.arcmin)
    df_bkg_all = ana_utils.select_centre(cat, coord1, params.r_cen * u.arcmin)
    df_cen_all = ana_utils.select_centre(cat, coord0, params.r_cen * u.arcmin)

    if fig is None:
        fig, axs = plt.subplots(2, 3, figsize=(6, 4))

    plt.sca(axs[0][0])
    tangent_axis()
    filt_no = ~np.isin(cat["INDEX"], cat_filtered["INDEX"])
    
    plt.scatter(cat[filt_no]["xi"], cat[filt_no]["eta"], s=1, lw=0, color="k")
    
    circ = plt.Circle((0, 0), params.r_cen, color="C0", lw=1, fill=False, zorder=-1, alpha=0.5)
    plt.gca().add_patch(circ)
    circ = plt.Circle((xi*60, eta*60), params.r_cen, color="C1", lw=1, fill=False, zorder=-1, alpha=0.5)
    plt.gca().add_patch(circ)
    
    
    plt.title("unselected")

    plt.sca(axs[1][0])
    tangent_axis()
    plt.scatter(cat_filtered["xi"], cat_filtered["eta"], s=1, lw=0, color="C0")
    
    circ = plt.Circle((0, 0), params.r_cen, color="C0", lw=1, fill=False, zorder=-1, alpha=0.5)
    plt.gca().add_patch(circ)
    circ = plt.Circle((xi*60, eta*60), params.r_cen, color="C1", lw=1, fill=False, zorder=-1, alpha=0.5)
    plt.gca().add_patch(circ)
    
    plt.title("selected")

    plt.sca(axs[1][1])
    gr_axis()
    plot_iso_gr(params)
    
    x = df_bkg_all[params.gcol] - df_bkg_all[params.rcol]
    y = df_bkg_all[params.gcol]
    plt.scatter(x, y, lw=0, s=1, color="k")
    
    x = df_bkg[params.gcol] - df_bkg[params.rcol]
    y = df_bkg[params.gcol]
    plt.scatter(x, y, lw=0, s=3, color="C0")
    plt.text(0.1, 0.9, f"${results.Nbkg_med:0.2f} \\pm {results.Nbkg_std:0.2f}$", transform=plt.gca().transAxes, color="C1", fontsize=8)

    plt.title("background")

    
    plt.sca(axs[0][1])
    gr_axis()
    plot_iso_gr(params)

    x = df_cen_all[params.gcol] - df_cen_all[params.rcol]
    y = df_cen_all[params.gcol]
    plt.scatter(x, y, lw=0, s=1, color="k")

    x = df_cen[params.gcol] - df_cen[params.rcol]
    y = df_cen[params.gcol]
    plt.scatter(x, y, lw=0, s=3, color="C0")
    plt.text(0.1, 0.9, results.Ncen, transform=plt.gca().transAxes, fontsize=8, color="C1")

    plt.title("centre")

    
    plt.sca(axs[1][2])
    ri_axis()
    plot_iso_ri(params)
    x = df_bkg_all[params.rcol] - df_bkg_all[params.icol]
    y = df_bkg_all[params.rcol]
    plt.scatter(x, y, lw=0, s=1, color="k")
    
    x = df_bkg[params.rcol] - df_bkg[params.icol]
    y = df_bkg[params.rcol]
    plt.scatter(x, y, lw=0, s=3, color="C0")
    plt.title("background")


    plt.sca(axs[0][2])
    ri_axis()
    plot_iso_ri(params)
    
    x = df_cen_all[params.rcol] - df_cen_all[params.icol]
    y = df_cen_all[params.rcol]
    plt.scatter(x, y, lw=0, s=1, color="k")

    x = df_cen[params.rcol] - df_cen[params.icol]
    y = df_cen[params.rcol]
    plt.scatter(x, y, lw=0, s=3, color="C0")
    
    plt.title("centre")

    # plot_ri_depth()

    plt.tight_layout()

In [ ]:
def tangent_axis():
    plt.ylabel(r"$\eta/\textrm{arcmin}$")
    plt.xlabel(r"$\xi/\textrm{arcmin}$")
    plt.xticks(np.arange(-5, 6))
    plt.yticks(np.arange(-5, 6))

    plt.xlim(tangent_bounds[2], tangent_bounds[0])
    plt.ylim(tangent_bounds[1], tangent_bounds[3])
    plt.gca().set_aspect(1)

In [ ]:
def ri_axis():
    plt.xlabel(r"$r-i$ (mag)")
    plt.ylabel(r"$r$ (mag)")

    plt.xlim(-1.2, 1.8)
    plt.ylim(27, 17)
    plt.gca().set_box_aspect(1)


def gr_axis():
    plt.xlabel(r"$g-r$ (mag)")
    plt.ylabel(r"$g$ (mag)")
    plt.xlim(-1, 2)
    plt.ylim(27, 17)
    plt.gca().set_box_aspect(1)


In [ ]:
def plot_iso(iso, dm, A_b, A_r, mag_err, phase_max=3, b="SDSS_g", r="SDSS_r", iso_width=0):
    filt = iso["phase"] < phase_max
    plt.plot(iso[b][filt] - iso[r][filt] + A_b-A_r, iso[b][filt] + dm+A_b, color="grey", zorder=-1)

    x_poly, y_poly = make_polygon(iso, mag_err, dm=dm, A_b=A_b, A_r=A_r, b=b, r=r, iso_width=iso_width)
    plt.plot(x_poly, y_poly, color="grey", alpha=0.5, zorder=-1)


In [ ]:
def plot_iso_gr(params):
    A_g, A_r, A_i = get_extinction(params)
    iso = isochrones[params.iso_fe_h][params.iso_log_age]
    plot_iso(iso, dm=params.dm, A_b=A_g, A_r=A_r, mag_err=gr_err, iso_width=params.iso_width)
    
def plot_iso_ri(params):
    A_g, A_r, A_i = get_extinction(params)
    iso = isochrones[params.iso_fe_h][params.iso_log_age]

    plot_iso(iso, dm=params.dm, A_b=A_r, A_r=A_i, mag_err=ri_err, b="SDSS_r", r="SDSS_i", iso_width=params.iso_width)


In [ ]:
params_ri = filter_params(mode="ri")
params_ri_sgs = filter_params(mode="ri", r23_max_sigma=3)
params_gr = filter_params(mode="gr")
params_gri = filter_params(mode="gri")
params_n = filter_params(mode="n")

In [ ]:
1 / catalog_stats(cat, params_n).area_scale

In [ ]:
plot_2d_counts(cat, 20/60, thresh=36)

In [ ]:
plot_with_hist(cat, params_n)

The plot above is our detection diagnostic plot for an unfiltered sample. Each panel (in reading order) is
1. Top left: the stars excluded by the present selection criteria (black) on the tangent plane. The blue and orange circles represent the example central and background regions.
2. Top middle left: The g-r CMD for stars within the central region. Blue pointss are selected by the filter criterion and black are not.
3. Top middle right: Same as 2 but for r-i
4. Top right: The probability distribution of the counts of selected stars within background regions. The blue histogram is from randomly selected regions, masking those which are near the edge, centre, or bright stars. The orange histogram is instead from circles placed evenly (without filtering) on the grid across the shown tangent plane. The p-values are the quantiles of the central counts (verticle line) for the blue and orange histogram cases.

In [ ]:
plot_with_hist(cat, params_gr)

In [ ]:
plot_with_hist(cat, params_ri)

In [ ]:
plot_with_hist(cat, params_ri_sgs)

In [ ]:
plot_with_hist(cat, params_gri)

In [ ]:
dm_0 = obs_props["dm"]

for delta_dm in np.arange(-1, 1.5, 0.2):
    dm = dm_0 + delta_dm
    params = filter_params(dm=dm, mode="gri")
    # cat_filt = filter_catalog(cat, params)
    # plot_2d_counts(cat_filt, params.r_cen)
    plot_with_hist(cat, params)
    plt.gcf().suptitle(f"distance modulus = {dm:0.2f}")
    plt.show()

In [ ]:
ana_utils.get_extinction(0.2)

In [ ]:
A_V_0 = 0

for delta_A_V in np.arange(-0.2, 0.3, 0.05):
    A_V = A_V_0 + delta_A_V
    params = filter_params(A_V=A_V)
    # cat_filt = filter_catalog(cat, params)
    # plot_2d_counts(cat_filt, params.r_cen, vmax=40)

    plot_with_hist(cat, params)

    plt.gcf().suptitle(f"A_V = {A_V:0.2f}")
    plt.show()

In [ ]:


for col in mag_choices:
    params = filter_params(
        gcol = f"G_{col}",
        rcol = f"R_{col}",
        icol = f"I_{col}",
    )
    # cat_filt = filter_catalog(cat, params)
    # plot_2d_counts(cat_filt, params.r_cen, vmax=40)

    plot_with_hist(cat, params)

    plt.gcf().suptitle(f"magnitude = {col}")
    plt.show()

In [ ]:
for iso_fe_h in isos_fe_h:
    params = filter_params(iso_fe_h = iso_fe_h)


    plot_with_hist(cat, params)

    plt.gcf().suptitle(f"iso fe = {iso_fe_h}")
    plt.show()

In [ ]:
for iso_age in np.arange(9.7, 10.3, 0.1):
    params = filter_params(iso_log_age = iso_age)


    plot_with_hist(cat, params)

    plt.gcf().suptitle(f"iso log age = {iso_age}")
    plt.show()

In [ ]:
for star_gal_mode in ["none", "ell", "fwhm", "r_half", "r23"]:

    kwargs = {
        "none": dict(),
        "ell": dict(ellipticity_max = 0.2),
        "fwhm": dict(fwhm_max = 6),
        "r_half": dict(r_half_max = 6),
        "r23": dict(r23_max_sigma = 3)
    }
    
    params = filter_params(**kwargs[star_gal_mode])


    plot_with_hist(cat, params)

    plt.gcf().suptitle(f"star galaxy sep = {star_gal_mode}")
    plt.show()

# Running the MCMC samples

In [ ]:
df = pd.DataFrame()

for i in range(300):

    if i % 20 == 0:
        print("step ", i)

    col = np.random.choice(mag_choices)
    params = filter_params(
        detection_sigma = np.random.choice([1.5, 2, 2.5, 3, 6]),
        r_cen = np.random.uniform(10, 80)/60,
        A_V = obs_props["A_V"] + np.random.normal(0, 0.2),
        dm = obs_props["dm"] + np.random.normal(0, 0.2),
        flags_max = np.random.choice([0, 4, 8, 1024]),
        flags_weight_max = np.random.choice([0, 2, 1024]),
        iso_fe_h = np.random.choice(isos_fe_h),
        iso_log_age = np.random.choice(np.arange(9.4, 10.3, 0.1)),
        iso_width = np.random.uniform(0, 0.1),
        mode = np.random.choice([ "ri",  "gr", "n"]),
        gcol = f"G_{col}",
        rcol = f"R_{col}",
        icol = f"I_{col}",
        A_i_extra = np.random.normal(0, 0.15),
    )
    d = catalog_stats(cat, params)

    df = pd.concat([df, d], ignore_index=True)

In [ ]:
df["significance"] = df["delta"] / df["delta_err"]
df.loc[~np.isfinite(df["significance"]), "significance"] = 0

In [ ]:
plt.scatter(df["significance"], df["sigma"] / df["sigma_err"], c=np.log10(df["Ncen"]), s=2, lw=0)
plt.axhline(2)
plt.axvline(2)
plt.xlabel("significance (bootstrap)")
plt.ylabel("significance (poisson/mean)")
plt.colorbar(label="log num members")

In [ ]:
plt.scatter(df["significance"], df["pval"], c=np.log10(df["delta"]))

plt.yscale('log')
plt.gca().invert_yaxis()

plt.ylabel("pvalue")
plt.xlabel("significance")
plt.colorbar(label="delta (overdensity)")

In [ ]:
plt.scatter(df["Ncen"], df["significance"], c=np.log10(df["pval"]))

plt.xscale('log')

plt.ylabel("significance")
plt.xlabel("number of stars in centre")
plt.colorbar(label="pvalue")

In [ ]:
for col in df.columns:
    if col not in ["sigma", "sigma_err", "Ncen", "Ntot", "Nbkg_med", "Nbkg_std", "pval", "delta", "delta_err", "significance"]:
        plt.figure(figsize=(4, 2))
        if len(df[col].unique()) == 1:
            continue
        elif len(df[col].unique()) < 20:
            groups = df.groupby(col).groups
            plt.boxplot([df.iloc[idx]["significance"] for idx in groups.values()], tick_labels=groups.keys(), )
            plt.xlabel(col)
            plt.ylabel("significance")
            plt.xticks(rotation=90)

        else:
            plt.scatter(df[col], df["significance"], c=-np.log10(df["pval"]), s=3)
            plt.xlabel(col)
            plt.ylabel("significance")
        # plt.gca().invert_yaxis()
        # plt.yscale("log")
            plt.colorbar(label="-log pvalue")
        plt.show()

In [ ]:
def retrieve_params(df_row):
    kwargs = {key: df_row[key] for key in filter_params.__dataclass_fields__.keys()}

    return filter_params(**kwargs)

In [ ]:
df.sort_values("significance")

In [ ]:
plt.hist(np.log10(df["pval"][df.pval > 0]))

In [ ]:
plt.scatter(np.sqrt(df.Ntot) * df.area_scale, df.Nbkg_std)
plt.plot([0, 20], [0, 20], color="k")

In [ ]:
plt.scatter(df.Ntot_scaled, df.Nbkg_med)
plt.plot([0, 300], [0, 300], color="k")

In [ ]:
plt.hist(df.significance, bins=np.linspace(-3, 5, 40), histtype="step", density=True, label="bootstrap")

plt.hist(df.sigma / df.sigma_err, bins=np.linspace(-3, 5, 40), histtype="step", density=True, label="average + poisson")
plt.xlabel("significance")
plt.ylabel("density")

plt.legend()

In [ ]:
params_best = retrieve_params(df.iloc[np.argmax(df.significance)])

In [ ]:
params_best

In [ ]:
catalog_stats(cat, params_best)

# Better plots

In [ ]:
from astropy.coordinates import SkyCoord

In [ ]:
params_best

In [ ]:
df_sorted = df.sort_values("delta")[::-1]
df_sorted = df_sorted[np.isfinite(df.delta)]

In [ ]:
10**10.1

In [ ]:
df_sorted

The best examples

In [ ]:
for i in range(10):
    params = retrieve_params(df_sorted.iloc[i, :])
    plot_riso(cat, params, results=df_sorted.iloc[i, :])


The worst examples

In [ ]:
for i in range(10):
    d = df_sorted.iloc[-i-1, :]
    params = retrieve_params(d)
    plot_riso(cat, params, results=d)

# Sanity checks

In [ ]:
samples = [rand_overdensity(cat, 40/60, xymax=3) for i in  range(10_000)]

In [ ]:
sample_sigma = [x[0] for x in samples]

In [ ]:
sample_err = [x[1] for x in samples]

In [ ]:

plt.hist(samples)
plt.axvline(inner_overdensity(cat, 40/60)[3], color="C1")
plt.xlabel("counts")

In [ ]:
plt.hist((sample_sigma) / np.array(sample_err), density=True)
x = np.linspace(-5, 5, 1000)
y = 1/np.sqrt(2*np.pi) * np.exp(-x**2 / 2)
plt.plot(x, y)

In [ ]:
cat_test = pd.DataFrame(
    dict(
        xi = np.random.uniform(-3, 3, 2000),
        eta = np.random.uniform(-3, 3, 2000)
    )
)